# Phase 2 — train the plate detector

Trains YOLO on the dataset Phase 1 built. Settings come from `configs/detector.yaml`, committed to the repo, so a result is reproducible from the repo rather than from whatever was typed into a cell.

**Requires a T4.** `Runtime > Change runtime type > T4 GPU`.

**Run `01_build_dataset.ipynb` first in this same session.** `data/yolo` lives on the Colab VM and does not survive a runtime restart.

In [ ]:
REPO = "https://github.com/fayazhussain2821/Automatic-License-Plate-Recognition.git"
BRANCH = "main"

import os

if os.path.isdir("/content/ALPR"):
    !cd /content/ALPR && git fetch --quiet origin && git checkout --quiet $BRANCH && git pull --quiet
else:
    !git clone --quiet --branch $BRANCH $REPO /content/ALPR

%cd /content/ALPR
!pip install --quiet -e .

# An editable install registers itself through a .pth file, which is only
# read at interpreter startup — a running kernel cannot see it otherwise.
import sys

SRC = "/content/ALPR/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import alpr

print(f"alpr {alpr.__version__} importable")

## 0. Make sure the dataset exists

`ensure_dataset()` is a no-op when this session already has the data, and a full rebuild (download → manifest → split → export) when it does not. That makes this notebook runnable on its own in a fresh runtime — which matters, because free Colab allows one session and `/content` does not survive it.

In [ ]:
from alpr.build import ensure_dataset

# Rebuilds the dataset if this session does not already have it, and returns
# immediately if it does. Colab's free tier gives one GPU session and
# everything under /content dies with it, so a notebook that depends on a
# previous notebook's output usually cannot run at all.
#
# Needs ROBOFLOW_API_KEY in the 🔑 Secrets panel only when a download is
# actually required.
DATA = ensure_dataset()
print(f"dataset ready: {DATA}")

## 1. Pre-flight

Both checks fail in seconds. Discovering a missing dataset or a CPU runtime *after* an hour of training is the expensive version of this.

In [ ]:
from alpr.env import require_gpu
from alpr.train import TrainConfig, validate_dataset

DATA = "data/yolo/data.yaml"

gpu = require_gpu()
print(f"gpu:   {gpu}")

spec = validate_dataset(DATA)
print(f"data:  {spec['names']}")

config = TrainConfig.from_yaml("configs/detector.yaml")
print(f"model: {config.model}  imgsz={config.imgsz}  epochs={config.epochs}  batch={config.batch}")

## 2. Train

Expect roughly 40–70 minutes for 100 epochs on ~2,200 training images at 640px on a T4. Training stops early if 25 epochs pass without improvement.

If the session drops, set `resume: true` in `configs/detector.yaml` and rerun — Ultralytics picks up from the last checkpoint.

In [ ]:
from alpr.train import train

results = train(config, DATA)

## 3. Results

Phase 3 gates on **mAP@50 ≥ 0.85** against the held-out test split. What prints here is validation, measured during training — treat it as an early read, not the gate.

In [ ]:
from alpr.train import best_weights

weights = best_weights(config)
print(f"weights: {weights}  ({weights.stat().st_size / 1e6:.1f} MB)")

metrics = results.results_dict if hasattr(results, "results_dict") else {}
for key, value in metrics.items():
    print(f"{key:<28} {value:.4f}" if isinstance(value, float) else f"{key:<28} {value}")

In [ ]:
from IPython.display import Image, display

run_dir = weights.parent.parent
for plot in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    path = run_dir / plot
    if path.exists():
        print(plot)
        display(Image(filename=str(path), width=900))

## 4. Save the weights off the VM

**Do this before the session ends.** `runs/` dies with the runtime, and an hour of T4 time dies with it.

`provenance.json` travels alongside the weights: it records the config *and* the Ultralytics version, which changes augmentation defaults between releases. Without it a number cannot be reproduced later.

In [ ]:
# Option A — Hugging Face Hub. Needs HF_TOKEN in the 🔑 Secrets panel.
#
# from huggingface_hub import HfApi
# from alpr.env import get_credential
#
# api = HfApi(token=get_credential("HF_TOKEN"))
# repo_id = "Babblu2821/alpr-plate-detector"
# api.create_repo(repo_id, exist_ok=True)
# api.upload_file(path_or_fileobj=str(weights), path_in_repo="best.pt", repo_id=repo_id)
# api.upload_file(
#     path_or_fileobj=str(run_dir / "provenance.json"),
#     path_in_repo="provenance.json",
#     repo_id=repo_id,
# )

# Option B — download to your machine.
from google.colab import files

files.download(str(weights))